# Indic Parler-TTS — Maithili TTS Evaluation

This notebook generates speech for 20 phonetically balanced Maithili sentences using the **Indic Parler-TTS** model,
then evaluates intelligibility via Whisper ASR (WER & CER).

**Model:** [ai4bharat/indic-parler-tts](https://huggingface.co/ai4bharat/indic-parler-tts)
**Language:** Maithili (mai)
**Architecture:** Encoder-Decoder Transformer with DAC audio codec


In [ ]:
!pip install torch transformers accelerate soundfile scipy jiwer torchaudio
!pip install git+https://github.com/huggingface/parler-tts.git

In [ ]:
pip install --upgrade protobuf

In [ ]:
from huggingface_hub import login

login(token="your_hf_token_here")

In [ ]:
import json
import os
import shutil
import torch
import torchaudio
import soundfile as sf
from google.colab import files

from transformers import (
    AutoTokenizer,
    WhisperProcessor, WhisperForConditionalGeneration
)
from parler_tts import ParlerTTSForConditionalGeneration
from jiwer import wer, cer

# ==========================================
# 1. SETUP & DIRECTORIES
# ==========================================
device = "cuda:0" if torch.cuda.is_available() else "cpu"
output_dir = "parler_maithili_outputs"
os.makedirs(output_dir, exist_ok=True)

with open('maithili_balanced_set.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

if isinstance(raw_data, dict):
    for key, value in raw_data.items():
        if isinstance(value, list):
            raw_data = value
            break

sentences = []
for item in raw_data:
    if isinstance(item, dict):
        text_keys = ['text', 'sentence', 'transcript', 'transcription', 'content']
        found = False
        for k in text_keys:
            if k in item:
                sentences.append(str(item[k]))
                found = True
                break
        if not found:
            for val in item.values():
                if isinstance(val, str):
                    sentences.append(val)
                    break
    elif isinstance(item, str):
        sentences.append(item)

print(f"Loaded {len(sentences)} sentences for evaluation.\n")
if len(sentences) > 0:
    print(f"Sample: {sentences[0]}\n")

# ==========================================
# 2. LOAD MODELS
# ==========================================
print("Loading Indic Parler TTS Model...")
parler_id = "ai4bharat/indic-parler-tts"
parler_model = ParlerTTSForConditionalGeneration.from_pretrained(parler_id).to(device)
parler_tokenizer = AutoTokenizer.from_pretrained(parler_id)

print("Loading Whisper Medium ASR Model...")
asr_id = "openai/whisper-medium"
asr_processor = WhisperProcessor.from_pretrained(asr_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_id).to(device)

forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language="hindi", task="transcribe")

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def transcribe_audio_whisper(audio_path):
    """Loads audio, ensures 16kHz for Whisper, and transcribes."""
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)

    input_features = asr_processor(
        waveform.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids
        )

    transcription = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

# ==========================================
# 4. GENERATION & EVALUATION LOOP
# ==========================================
results = []
voice_condition = "A clear and expressive female speaker delivering a speech in a high-quality studio environment."

for i, text in enumerate(sentences):
    print(f"Processing Sentence {i+1}/{len(sentences)}...")
    text = str(text)
    parler_path = os.path.join(output_dir, f"parler_sent_{i+1}.wav")

    # --- Generate Parler ---
    cond_inputs = parler_tokenizer(voice_condition, return_tensors="pt").input_ids.to(device).long()
    prompt_inputs = parler_tokenizer(text, return_tensors="pt").input_ids.to(device).long()

    with torch.no_grad():
        parler_out = parler_model.generate(input_ids=cond_inputs, prompt_input_ids=prompt_inputs)
    parler_audio = parler_out.cpu().numpy().squeeze()
    sf.write(parler_path, parler_audio, parler_model.config.sampling_rate)

    # Transcribe Parler
    parler_transcription = transcribe_audio_whisper(parler_path)
    parler_wer = wer(text, parler_transcription)
    parler_cer = cer(text, parler_transcription)

    results.append({
        "id": i + 1,
        "original_text": text,
        "transcription": parler_transcription,
        "wer": parler_wer,
        "cer": parler_cer
    })

# ==========================================
# 5. SAVE METRICS & ZIP ARCHIVE
# ==========================================
with open(os.path.join(output_dir, "evaluation_metrics.json"), 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("\nEvaluation Complete. Zipping files...")
zip_filename = "indic_parler_maithili_audio"
shutil.make_archive(zip_filename, 'zip', output_dir)

print("Triggering download...")
files.download(f"{zip_filename}.zip")

